# 1- Transfer Learning Setup

In [7]:
import tensorflow as tf
from keras import layers, Model
from keras.applications import EfficientNetB0
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import preprocessing

NUM_CLASSES = 4  # plastic, metal, glass, cardboard
IMG_SIZE = (224, 224)

def build_model(num_classes, trainable_base=False):

    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",       
        input_shape=(224, 224, 3)
    )
    base_model.trainable = trainable_base  # Freeze during Phase 1

    inputs = tf.keras.Input(shape=(224, 224, 3))
    
    x = base_model(inputs, training=False)  # training=False keeps BatchNorm frozen

    # Custom head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)               # Regularization
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(NUM_CLASSES)
model.summary()

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data/raw'

# 2- Train head only (base frozen)

In [ ]:
import preprocessing as preprocessing


model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_p1 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    ReduceLROnPlateau(factor=0.5, patience=3, monitor="val_loss", verbose=1),
    ModelCheckpoint("best_phase1.keras", save_best_only=True, monitor="val_accuracy")
]

print("\n=== Phase 1: Training head (base frozen) ===")
history1 = model.fit(
    preprocessing.train_generator,
    epochs=20,
    validation_data=preprocessing.val_generator,
    callbacks=callbacks_p1
)
